# Data Unit Testing — Car Price Dataset

Adapted from the class Great Expectations exercise (`01_Data_Unit_Tests.ipynb`), applied to our MLOps project data: used-car listings with the goal of predicting `price`.

## Why this matters for our project
A quick profile of `train.csv` revealed real data quality problems:
- Categorical columns (`Brand`, `transmission`, `fuelType`) have casing, whitespace, truncation, and typo variants (e.g. `'VW'`, `'vw'`, `'w'`, `'W'` all mean the same brand).
- Numeric columns that should never be negative (`mileage`, `tax`, `mpg`, `engineSize`, `previousOwners`) contain negative values.
- `paintQuality%` exceeds 100 in places.
- `year` contains implausible values (as old as 1970, and fractional/future years like 2024.12).
- `hasDamage` only ever contains `0` or null — never `1` — which is itself suspicious and worth flagging as an expectation (and a modelling caveat).
- Several columns have 2-10% missing values.

These are exactly the kinds of issues a Data Unit Testing step should catch automatically, before they ever reach the cleaning/feature-engineering pipeline.

In [1]:
import pandas as pd
import great_expectations as gx
import warnings
warnings.filterwarnings('ignore')

print(f"Great Expectations Version: {gx.__version__}")

Great Expectations Version: 1.18.1


## Step 1: Load reference vs. incoming batch

In a real pipeline, `train.csv` is the historical reference data we trust to define our contract, and `test.csv` (or any new scrape of listings) is the incoming batch we validate before it's allowed into the pipeline.

In [2]:
df_reference = pd.read_csv("../data/01_raw/train.csv")
df_batch = pd.read_csv("../data/01_raw/test.csv")

print(f"Reference (train) shape: {df_reference.shape}")
print(f"Batch (test) shape: {df_batch.shape}")

Reference (train) shape: (75973, 14)
Batch (test) shape: (32567, 13)


## Step 2: Initialize the Data Context

In [3]:
context = gx.get_context(project_root_dir="gx_project")
print("Context initialized successfully!")

Context initialized successfully!


## Step 3: Connect data & build the Expectation Suite

Rules encode what "good" car listing data looks like, based on domain knowledge plus the profiling above:

1. **Identity**: `carID` must be unique and never null.
2. **Categorical integrity**: `transmission` and `fuelType`, *once normalized* (lowercased/stripped), should fall within a known set. (Note: raw values are messy —> normalization note.)
3. **Numeric boundaries**:
   - `year` between 1990 and 2026 (current year)
   - `price` greater than 0
   - `mileage` >= 0
   - `tax` >= 0
   - `mpg` >= 0
   - `engineSize` between 0 and 8.0 (litres)
   - `paintQuality%` between 0 and 100
   - `previousOwners` between 0 and 15
4. **Completeness**: `price` (target, train only) must never be null.

**Important note on normalization**: because `Brand`/`transmission`/`fuelType` have casing/whitespace/typo variants, a strict `ExpectColumnDistinctValuesToBeInSet` on the *raw* column will fail constantly. We define two suites:
- `car_quality_raw_v1`: loose checks on raw data (nulls, numeric ranges) — this is what should gate ingestion.
- We leave categorical *normalization* to the `data_cleaning` pipeline, not data quality gating, since messy categoricals are an expected/fixable condition, not a reason to reject the batch outright.

In [4]:
datasource_name = "car_price_data_source"
try:
    car_datasource = context.data_sources.get(datasource_name)
except Exception:
    car_datasource = context.data_sources.add_pandas(name=datasource_name)

try:
    asset_ref = car_datasource.get_asset("asset_reference")
    asset_batch = car_datasource.get_asset("asset_batch")
except Exception:
    asset_ref = car_datasource.add_dataframe_asset(name="asset_reference")
    asset_batch = car_datasource.add_dataframe_asset(name="asset_batch")

try:
    batch_def_ref = asset_ref.get_batch_definition("batch_definition_ref")
    batch_def_batch = asset_batch.get_batch_definition("batch_definition_batch")
except Exception:
    batch_def_ref = asset_ref.add_batch_definition_whole_dataframe("batch_definition_ref")
    batch_def_batch = asset_batch.add_batch_definition_whole_dataframe("batch_definition_batch")

suite_name = "car_quality_raw_v1"
try:
    context.suites.delete(suite_name)
except Exception:
    pass

suite = gx.core.expectation_suite.ExpectationSuite(name=suite_name)

# Identity
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeUnique(column="carID"))
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="carID"))

# Numeric boundaries (allow nulls here -- missingness is handled separately, not a hard fail)
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column="year", min_value=1990, max_value=2026, mostly=0.99))
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column="mileage", min_value=0, max_value=None, mostly=0.95))
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column="tax", min_value=0, max_value=None, mostly=0.95))
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column="mpg", min_value=0, max_value=None, mostly=0.95))
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column="engineSize", min_value=0, max_value=8.0, mostly=0.95))
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column="paintQuality%", min_value=0, max_value=100, mostly=0.95))
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column="previousOwners", min_value=0, max_value=15, mostly=0.95))
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeInSet(column="hasDamage", value_set=[0, 1], mostly=0.95))

context.suites.add(suite)
print(f"✅ Expectation Suite '{suite_name}' successfully created and registered!")

✅ Expectation Suite 'car_quality_raw_v1' successfully created and registered!


## Step 4: Run validation on the incoming batch (test.csv)

In [5]:
validation_name = "car_batch_validation"
FORCE_RECREATE = True

if FORCE_RECREATE:
    try:
        context.validation_definitions.delete(validation_name)
    except Exception:
        pass

try:
    validation_definition = context.validation_definitions.get(validation_name)
except Exception:
    validation_definition = context.validation_definitions.add(
        gx.core.validation_definition.ValidationDefinition(
            name=validation_name,
            data=batch_def_batch,
            suite=suite,
        )
    )

validation_results = validation_definition.run(batch_parameters={"dataframe": df_batch})

if validation_results.success:
    print("✅ Validation Passed! Proceeding to cleaning/feature engineering...")
else:
    print("🚨 Validation Failed! Review failing expectations below before proceeding.")

Calculating Metrics:   0%|          | 0/72 [00:00<?, ?it/s]

✅ Validation Passed! Proceeding to cleaning/feature engineering...


In [6]:
def get_validation_results(validation_results):
    """Convert the GX object to a dictionary for easy parsing.

    Args:
        validation_results (_type_): _description_

    Returns:
        Dataframe: _description_
    """
    validation_dict = validation_results.to_json_dict() if hasattr(validation_results, "to_json_dict") else validation_results
    results = validation_dict.get("results", [])
    rows = []
    for result in results:
        kwargs = result.get('expectation_config', {}).get('kwargs', {})
        res = result.get('result', {})
        rows.append({
            "Success": result.get('success', ''),
            "Expectation Type": result.get('expectation_config', {}).get('expectation_type', ''),
            "Column": kwargs.get('column', ''),
            "Min Value": kwargs.get('min_value', ''),
            "Max Value": kwargs.get('max_value', ''),
            "Element Count": res.get('element_count', ''),
            "Unexpected Count": res.get('unexpected_count', ''),
            "Unexpected Percent": res.get('unexpected_percent', ''),
        })
    return pd.DataFrame(rows)

df_test_results = get_validation_results(validation_results)
df_test_results

,Success,Expectation Type,Column,Min Value,Max Value,Element Count,Unexpected Count,Unexpected Percent
0,True,,carID,,,32567,0,0.000000
1,True,,carID,,,32567,0,0.000000
2,True,,year,1990.0,2026.0,32567,0,0.000000
3,True,,mileage,0.0,,32567,170,0.533283
4,True,,tax,0.0,,32567,161,0.550258
5,True,,mpg,0.0,,32567,17,0.058062
6,True,,engineSize,0.0,8.0,32567,33,0.103322
7,True,,paintQuality%,0.0,100.0,32567,168,0.525953
8,True,,previousOwners,0.0,15.0,32567,168,0.525493
9,True,,hasDamage,,,32567,0,0.000000


## Step 5: Profile the categorical mess 

Exploratory evidence for the report, showing *why* the cleaning pipeline needs a normalization step.

In [7]:
for col in ["Brand", "transmission", "fuelType"]:
    n_unique_raw = df_reference[col].nunique()
    n_unique_normalized = df_reference[col].str.strip().str.lower().nunique()
    print(f"{col}: {n_unique_raw} raw unique values -> {n_unique_normalized} after normalization (still includes typos/truncations)")

Brand: 72 raw unique values -> 33 after normalization (still includes typos/truncations)
transmission: 40 raw unique values -> 17 after normalization (still includes typos/truncations)
fuelType: 34 raw unique values -> 16 after normalization (still includes typos/truncations)


## Step 6: Auto-Generating an Expectation Suite with YData Profiling

Steps 1-4 hand-picked 9 rules based on domain knowledge — manageable for ~13 columns, but it doesn't scale, and it relies on us already knowing what "good" looks like. `ydata_profiling` takes the opposite approach: scan `train.csv` once, compute statistical bounds for every column automatically, and convert those bounds straight into a Great Expectations suite.

This matters here specifically because our manual suite only checked the columns we already suspected were problematic (`mileage`, `tax`, `paintQuality%`, etc.). The auto-suite covers `carID`, `Brand`, `model`, `transmission`, `fuelType` and every other column too — including ones we didn't think to write rules for, like exact value-set membership on `Brand` or `transmission` after cleaning. It's a useful second pass to catch anything Steps 1-4 missed.

The tradeoff is strictness: where our manual suite used `mostly=0.95` (allow 5% noise), the auto-suite locks numeric columns to their *exact* observed min/max/mean/median in the reference data. That makes it a tight drift detector — useful for catching when a new batch's distribution has shifted — but also means it will fail on differences that the manual suite would happily tolerate. We build it on the *cleaned* reference data, not the raw `train.csv`, so it learns sane bounds rather than the negative-mileage/`paintQuality%`-over-100 noise we already know is in there.

What we'll do:
1. Profile the cleaned reference data with `ProfileReport`.
2. Patch YData's expectation-generation logic to work with Great Expectations ≥ 1.0 (the built-in converter targets an older GX API).
3. Auto-generate a new suite, `car_auto_generated_suite`, from the profile.
4. Validate the cleaned batch (`test.csv`) against it and compare results with the manual suite from Steps 1-4.

In [8]:
from ydata_profiling import ProfileReport
import warnings
warnings.filterwarnings('ignore')

print("YData Profiling imported successfully")

YData Profiling imported successfully


In [9]:
# Build the reference profile on a cleaned version of train.csv so the auto-suite's
# bounds reflect sane values, not the raw data's negative/out-of-range noise.
# (clean_car_data is the same function used in the Kedro data_feat_engineering pipeline)
import importlib.util
spec = importlib.util.spec_from_file_location(
    "car_nodes", "../src/car_price_mlops/pipelines/data_feat_engineering/nodes.py"
)
car_nodes = importlib.util.module_from_spec(spec)
spec.loader.exec_module(car_nodes)
clean_car_data = car_nodes.clean_car_data

df_reference_clean = clean_car_data(df_reference)

profile = ProfileReport(df_reference_clean, title="Car Price Reference Profiling", minimal=True)
profile

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 14/14 [00:00<00:00, 377.91it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

#### GX adapter classes (converts profile output into expectations)

In [10]:
from typing import Any, Optional
import pandas as pd
from visions import VisionsTypeset

from ydata_profiling.config import Settings
from ydata_profiling.model import BaseDescription, expectation_algorithms
from ydata_profiling.model.handler import Handler
from ydata_profiling.utils.dataframe import slugify


def patched_numeric_expectations(name: str, summary: dict, batch: Any, *args):
    numeric_type_names = ["integer", "int", "int_", "int8", "int16", "int32", "int64", "uint8", "uint16", "uint32", "uint64", "float", "float_", "float16", "float32", "float64"]
    batch.expect_column_values_to_be_in_type_list(column=name, type_list=numeric_type_names)

    if "min" in summary: batch.expect_column_min_to_be_between(column=name, min_value=summary["min"], max_value=summary["min"])
    if "max" in summary: batch.expect_column_max_to_be_between(column=name, min_value=summary["max"], max_value=summary["max"])
    if "mean" in summary: batch.expect_column_mean_to_be_between(column=name, min_value=summary["mean"], max_value=summary["mean"])
    if "50%" in summary: batch.expect_column_median_to_be_between(column=name, min_value=summary["50%"], max_value=summary["50%"])
    return name, summary, batch

def patched_categorical_expectations(name: str, summary: dict, batch: Any, *args):
    string_and_bool_types = ["string", "str", "object", "O", "bool", "boolean"]
    batch.expect_column_values_to_be_in_type_list(column=name, type_list=string_and_bool_types)

    if "value_counts_without_nan" in summary:
        value_set = list(summary["value_counts_without_nan"].keys())
        batch.expect_column_values_to_be_in_set(column=name, value_set=value_set)
    return name, summary, batch

def patched_datetime_expectations(name: str, summary: dict, batch: Any, *args):
    datetime_types = ["datetime", "datetime64", "datetime64[ns]", "timedelta[ns]"]
    batch.expect_column_values_to_be_in_type_list(column=name, type_list=datetime_types)

    if "min" in summary: batch.expect_column_min_to_be_between(column=name, min_value=summary["min"], max_value=summary["min"])
    if "max" in summary: batch.expect_column_max_to_be_between(column=name, min_value=summary["max"], max_value=summary["max"])
    return name, summary, batch


class ExpectationHandler(Handler):
    def __init__(self, typeset: VisionsTypeset, *args, **kwargs):
        mapping = {
            "Unsupported": [expectation_algorithms.generic_expectations],
            "URL": [expectation_algorithms.url_expectations],
            "File": [expectation_algorithms.file_expectations],
            "Path": [expectation_algorithms.path_expectations],
            "Image": [expectation_algorithms.image_expectations],
            "Text": [expectation_algorithms.generic_expectations, patched_categorical_expectations],
            "Categorical": [expectation_algorithms.generic_expectations, patched_categorical_expectations],
            "Boolean": [expectation_algorithms.generic_expectations, patched_categorical_expectations],
            "Numeric": [expectation_algorithms.generic_expectations, patched_numeric_expectations],
            "DateTime": [expectation_algorithms.generic_expectations, patched_datetime_expectations],
        }
        super().__init__(mapping, typeset, *args, **kwargs)


class ExpectationsReportV3:
    config: Settings
    df: Optional[pd.DataFrame] = None

    @property
    def typeset(self) -> Optional[VisionsTypeset]:
        return None

    def to_expectation_suite(
        self,
        datasource_name: str,
        data_asset_name: str,
        suite_name: Optional[str] = None,
        data_context: Optional[Any] = None,
        save_suite: bool = True,
        run_validation: bool = False,
        build_data_docs: bool = False,
        handler: Optional[Handler] = None,
    ) -> Any:
        import great_expectations as ge

        if suite_name is None:
            suite_name = slugify(self.config.title)

        if handler is None:
            handler = ExpectationHandler(self.typeset)

        if not data_context:
            data_context = ge.get_context()

        try:
            data_asset = data_context.data_sources.get(datasource_name).get_asset(data_asset_name)
        except AttributeError:
            data_asset = data_context.get_datasource(datasource_name).get_asset(data_asset_name)

        try:
            batch_request = data_asset.build_batch_request(options={"dataframe": self.df})
        except Exception:
            batch_request = data_asset.build_batch_request()

        try:
            suite = data_context.suites.get(suite_name)
            data_context.suites.delete(suite_name)
        except Exception:
            pass

        suite = ge.core.expectation_suite.ExpectationSuite(name=suite_name)
        data_context.suites.add(suite)

        validator = data_context.get_validator(batch_request=batch_request, expectation_suite=suite)
        summary: BaseDescription = self.get_description()

        for name, variable_summary in summary.variables.items():
            handler.handle(variable_summary["type"], name, variable_summary, validator)

        suite = validator.get_expectation_suite(discard_failed_expectations=False)

        if save_suite:
            try:
                data_context.suites.delete(suite_name)
            except Exception:
                pass
            data_context.suites.add(suite)

        return suite


from ydata_profiling.expectations_report import ExpectationsReport
ExpectationsReport.to_expectation_suite = ExpectationsReportV3.to_expectation_suite

print("YData algorithms adapted for GX > 1.0")

YData algorithms adapted for GX > 1.0


#### Generate the auto-suite

In [11]:
# Register the cleaned reference data as a new asset (separate from the raw 'asset_reference'
# used in Steps 1-4) so the auto-suite is built on sane bounds.
try:
    asset_ref_clean = car_datasource.get_asset("asset_reference_clean")
except Exception:
    asset_ref_clean = car_datasource.add_dataframe_asset(name="asset_reference_clean")

profile.df = df_reference_clean

auto_suite_name = "car_auto_generated_suite"
auto_suite = profile.to_expectation_suite(
    datasource_name="car_price_data_source",
    data_asset_name="asset_reference_clean",
    suite_name=auto_suite_name,
    data_context=context,
)
print(f"Auto-Suite '{auto_suite_name}' created with {len(auto_suite.expectations)} rules!")

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/4 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/4 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/4 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/4 [00:00<?, ?it/s]

Auto-Suite 'car_auto_generated_suite' created with 62 rules!


#### Validate the batch

In [12]:
df_batch_clean = clean_car_data(df_batch.assign(price=0))  # dummy price so clean_car_data doesn't drop rows; test.csv has no real price column
df_batch_clean = df_batch_clean.drop(columns=["price"])

auto_validation_name = "car_auto_batch_validation"
try:
    context.validation_definitions.delete(auto_validation_name)
except Exception:
    pass

auto_validation_definition = context.validation_definitions.add(
    gx.core.validation_definition.ValidationDefinition(
        name=auto_validation_name,
        data=batch_def_batch,
        suite=auto_suite,
    )
)

print("Running auto-suite validation against the cleaned batch...")
auto_validation_results = auto_validation_definition.run(batch_parameters={"dataframe": df_batch_clean})

if auto_validation_results.success:
    print("Auto-suite validation PASSED.")
else:
    print("Auto-suite validation FAILED -- review results below. This can be expected: the auto-suite is strict (exact statistical match), so minor distribution differences between train and test will trip it.")

Running auto-suite validation against the cleaned batch...


Calculating Metrics:   0%|          | 0/104 [00:00<?, ?it/s]

Auto-suite validation FAILED -- review results below. This can be expected: the auto-suite is strict (exact statistical match), so minor distribution differences between train and test will trip it.


#### Results

In [13]:
df_auto_results = get_validation_results(auto_validation_results)
df_auto_results

,Success,Expectation Type,Column,Min Value,Max Value,Element Count,Unexpected Count,Unexpected Percent
0,False,,price,,,,,
1,False,,price,450.0,450.0,,,
2,False,,price,159999.0,159999.0,,,
3,False,,price,16881.889553,16881.889553,,,
4,False,,price,14699.0,14699.0,,,
...,...,...,...,...,...,...,...,...
57,True,,previousOwners,,,,,
58,True,,previousOwners,0.0,0.0,,,
59,True,,previousOwners,6.258371,6.258371,,,
60,False,,previousOwners,2.016256,2.016256,,,
